# A100 GRPO Demo — Bioresearch (Inference-Mirror, Judging Edition)

This notebook is the **demo-facing** GRPO training run, sized for **A100 40GB** and engineered around the hackathon's two weakest-typically axes:

- **Observable evidence of training progress (20%):**
  - **Live Trackio dashboard** that the judges can watch during the demo (auto-syncs to a HF Space if `HF_TOKEN` + `space_id` are set, so it persists after Colab dies).
  - **Baseline-vs-trained eval block** that runs the *exact same* held-out `task_id`s before training and after training, then renders a side-by-side table + sample rollout transcripts.
- **Reward coherent + meaningful inference improvement (10%):**
  - **Reward-distribution diagnostic** sampled before training so we can see the reward signal has variance (not stuck at 0 or 1) — without this the GRPO advantage collapses.
  - **`LabShapingCallback`** decomposes lab-task reward into *process* (per-step shaping) and *terminal* (final outcome) so the curves show whether training is learning to *think* better or just to *guess* better.

All heavy logic lives in [`training_core.py`](../training_core.py) and [`training_a100.py`](../training_a100.py). This notebook is a thin driver. The companion smoke test is [`tests/test_training_a100.py`](../tests/test_training_a100.py).

**Hardware target:** A100 40GB (80GB documented inline). Drop to T4 by switching to [`train_grpo_inference_mirror.ipynb`](train_grpo_inference_mirror.ipynb) — same training_core module, smaller defaults.

## 1 · Install dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.12" peft accelerate bitsandbytes
!pip install -q httpx fastapi uvicorn pydantic pandas matplotlib datasets scipy
!pip install -q trackio  # live dashboard; the pipeline still runs without it

## 2 · Clone the environment repo (Colab)

In [ ]:
import os, subprocess, sys

REPO_URL = os.environ.get(
    "BIORESEARCH_REPO_URL",
    "https://huggingface.co/spaces/anirudhchida/bioresearch",
)

if not os.path.isdir("bioresearch"):
    if "anirudhchida" in REPO_URL:
        raise RuntimeError(
            "Set BIORESEARCH_REPO_URL (or edit REPO_URL above) to your fork — "
            "the placeholder URL will 404 on clone."
        )
    subprocess.run(["git", "clone", REPO_URL, "bioresearch"], check=True)

sys.path.insert(0, "/content/bioresearch")
print("Setup complete")

## 3 · Boot the OpenEnv server

The reward loop talks HTTP to this subprocess. Current OpenEnv releases expose `/health`, `/schema`, `/docs`, and `/openapi.json` — the legacy `/info` is gone.

In [ ]:
import subprocess, time, httpx

server_proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/bioresearch",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

for _ in range(40):
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=1.0).status_code == 200:
            print("Server up")
            break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError("OpenEnv server failed to start")

## 4 · Wire up `training_core` + Trackio

Trackio init is wrapped — if the install in Cell 1 failed or the network blocks the HF Space, `setup_trackio` degrades gracefully to `report_to="none"` and the rest of the notebook still produces all the same artifacts (just no live dashboard). Set `HF_SPACE_ID` to a `username/space-name` you own to enable persistent sync.

In [ ]:
import os
import training_core
import training_a100
from training_core import (
    LEGACY_TASKS, LAB_TASKS, ALL_TASKS, DEFAULT_T4_TASKS,
    TRAIN_LAB_MAX_STEPS, EVAL_LAB_MAX_STEPS,
)

# Bump the in-training lab rollout budget on A100 — richer process reward
# than the T4 notebook's TRAIN_LAB_MAX_STEPS=4.
os.environ["TRAIN_LAB_MAX_STEPS"] = "8"
training_core.TRAIN_LAB_MAX_STEPS = 8
training_core.EVAL_LAB_MAX_STEPS = 12

SERVER_URL = "http://127.0.0.1:8000"
training_core.configure_env(SERVER_URL)

# Optional persistent Trackio dashboard. Leave HF_SPACE_ID unset for a
# local-only dashboard (still visible in Colab via the trackio CLI).
trackio_cfg = training_a100.setup_trackio(
    project="bioresearch-grpo-a100",
    run_name="qwen2p5-7b-mirror-200steps",
    space_id=os.environ.get("HF_SPACE_ID"),
    config={"model": "Qwen2.5-7B-Instruct-bnb-4bit", "lora_r": 32},
)
print("Trackio config:", trackio_cfg)

probe = training_core.env_reset(task_type="dna_classification").observation
print(f"Probe OK  task_id={probe.task_id}  question[:60]={probe.question[:60]!r}")

## 5 · Choose the task list

Default trains all 14 tasks — this is the demo run. The 6-task T4 subset and a custom subset are documented as commented lines.

In [ ]:
TASK_LIST = ALL_TASKS                       # 14 reward curves — full demo
# TASK_LIST = DEFAULT_T4_TASKS              # 6-task T4 fallback
# TASK_LIST = ["dna_reasoning", "clinical_diagnosis", "protein_hypothesis_lab"]

print("Training on:", TASK_LIST)
print("  legacy:", [t for t in TASK_LIST if t in LEGACY_TASKS])
print("  lab   :", [t for t in TASK_LIST if t in LAB_TASKS])

## 6 · Build the mixed-task dataset

`n_per_task=8` on A100 40GB (112 rows for all 14 tasks). Bump to `12` on 80GB for a richer dataset; reward curves still smooth out fine at 8 rows × 14 tasks × 8 generations = ~900 rollouts per epoch.

In [ ]:
N_PER_TASK = 8                              # 12 on A100 80GB
train_ds = training_core.build_mixed_dataset(TASK_LIST, n_per_task=N_PER_TASK, seed=42)
print(train_ds)
print("Sample row keys:", train_ds.column_names)

## 7 · Load Qwen2.5-7B + LoRA (r=32)

Unsloth + 4-bit + LoRA r=32 fits comfortably in A100 40GB. After loading we print VRAM headroom — if it's < 4 GB drop `num_generations` to 4 in Cell 11. On 80GB you can swap to `Qwen2.5-14B-Instruct-bnb-4bit`.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # A100 40GB primary
# MODEL_NAME = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit" # A100 80GB upgrade
# MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit" # T4 fallback
MAX_SEQ_LEN = 3072

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
tokenizer.padding_side = "left"
training_core.configure_model(model, tokenizer, max_new_tokens=384)

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM headroom after model load: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total")
print("Model + tokenizer loaded and registered with training_core")

## 8 · Baseline rollouts (BEFORE training)

Run the *untrained* policy on a fixed seed of 5 held-out `task_id`s per task. We capture the pool here and reuse it after training so the before/after compare on identical briefs.

In [ ]:
FastLanguageModel.for_inference(model)        # 2x faster greedy-ish gen

baseline_rows, eval_pool = training_a100.collect_rollouts_with_pool(
    TASK_LIST, n_per_task=5, seed=2026,
)
training_a100.save_rollouts(baseline_rows, "baseline_rollouts.json")

# Per-task summary so we have a number to point at on the slide.
import statistics
print(f"Baseline rollouts (n={len(baseline_rows)}):")
for task in TASK_LIST:
    rs = [r["reward"] for r in baseline_rows if r["task_type"] == task]
    if rs:
        print(f"  {task:30s}  mean={statistics.fmean(rs):.3f}  n={len(rs)}")

FastLanguageModel.for_training(model)         # back to training mode

## 9 · Reward-distribution diagnostic

Score a hand-curated bag of completions per task through the reward function. The point isn't to evaluate the model — the completions are canned. The point is to **prove to the judges** that the reward signal:

1. Spans a meaningful range (not all 0 or 1).
2. Reflects completion quality (correct answers get high reward, garbage gets low).
3. Has variance for GRPO's relative advantage to actually optimize against.

In [ ]:
import matplotlib.pyplot as plt

diag_df = training_a100.reward_distribution_diagnostic(TASK_LIST, n_samples_per_task=4)
print(diag_df)

fig, ax = plt.subplots(figsize=(12, 4))
tasks_in_order = [t for t in TASK_LIST if t in set(diag_df["task_type"])]
data = [diag_df[diag_df["task_type"] == t]["reward"].tolist() for t in tasks_in_order]
ax.boxplot(data, labels=tasks_in_order, vert=True, showmeans=True)
ax.set_ylabel("Reward (canned-completion bag)")
ax.set_title("Reward signal has variance — GRPO has something to optimize")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("reward_diagnostic.png", dpi=140)
plt.show()

## 10 · `GRPOConfig` + `GRPOTrainer` (Trackio-wired)

A100 40GB defaults: `num_generations=8`, `max_completion_length=768`, `max_steps=200`. Bump `max_steps` to 500 if you have a 2-hour slot — the curves typically keep climbing.

`LabShapingCallback` drains the per-step lab rollout log on every `on_log` event so the dashboard shows process vs terminal reward decomposition for each lab task.

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

reward_fns = [training_core.make_reward_fn(t) for t in TASK_LIST]
print("Reward functions wired:")
for fn in reward_fns:
    print(" -", fn.__name__)

_BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

grpo_config = GRPOConfig(
    output_dir="grpo_bioresearch_a100",
    learning_rate=3e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=8,
    max_prompt_length=1280,
    max_completion_length=768,
    max_steps=200,                     # bump to 500 if you have ~2 hours
    logging_steps=1,
    save_strategy="steps",
    save_steps=50,
    bf16=_BF16_OK,
    fp16=not _BF16_OK,
    beta=0.04,
    max_grad_norm=1.0,
    seed=42,
    **trackio_cfg,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_ds,
    reward_funcs=reward_fns,
    callbacks=[training_a100.make_lab_shaping_callback()],
)
print("Trainer ready")

## 11 · Train

In [ ]:
trainer.train()
training_a100.trackio_finish()  # noop if Trackio is not in use

## 12 · Trained rollouts (AFTER training)

Same `task_ids` as Cell 8 — `eval_pool` pins them. This is what makes the before/after table a *paired* comparison instead of two unpaired draws.

In [ ]:
FastLanguageModel.for_inference(model)

trained_rows = training_a100.collect_rollouts(
    TASK_LIST, n_per_task=5, seed=2026, task_id_pool=eval_pool,
)
training_a100.save_rollouts(trained_rows, "trained_rollouts.json")

import statistics
print(f"Trained rollouts (n={len(trained_rows)}):")
for task in TASK_LIST:
    rs = [r["reward"] for r in trained_rows if r["task_type"] == task]
    if rs:
        print(f"  {task:30s}  mean={statistics.fmean(rs):.3f}  n={len(rs)}")

## 13 · Before-vs-after table + bar chart

Headline judging artifact #1. Per-task delta in mean reward, sorted by biggest win first. The bar chart makes the wins obvious at a glance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

table = training_a100.before_after_table(baseline_rows, trained_rows)
print(table.to_string(index=False) if hasattr(table, "to_string") else table)

# Bar chart: delta per task, colored by sign.
if hasattr(table, "iloc"):
    tasks = table["task_type"].tolist()
    deltas = table["delta"].tolist()
    baselines = table["baseline_mean"].tolist()
    traineds = table["trained_mean"].tolist()
else:
    tasks = [r["task_type"] for r in table]
    deltas = [r["delta"] for r in table]
    baselines = [r["baseline_mean"] for r in table]
    traineds = [r["trained_mean"] for r in table]

x = np.arange(len(tasks))
fig, ax = plt.subplots(figsize=(13, 5))
width = 0.38
ax.bar(x - width/2, baselines, width, label="Baseline (untrained)")
ax.bar(x + width/2, traineds, width, label="After GRPO training")
for i, d in enumerate(deltas):
    color = "green" if d > 0 else ("red" if d < 0 else "gray")
    ax.annotate(f"{d:+.2f}", xy=(x[i], max(baselines[i], traineds[i]) + 0.02),
                ha="center", color=color, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(tasks, rotation=40, ha="right")
ax.set_ylabel("Mean reward")
ax.set_title("Before vs After GRPO — per-task mean reward (paired task_ids)")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("before_after_bar.png", dpi=140)
plt.show()

## 14 · Sample rollout transcripts (the headline demo artifact)

Headline judging artifact #2. For each task, the 2 `task_id`s with the largest positive delta are shown side-by-side: prompt → baseline completion + reward → trained completion + reward. Judges can literally read the qualitative jump.

In [ ]:
from IPython.display import Markdown, display

transcripts_md = training_a100.render_sample_transcripts(
    baseline_rows, trained_rows, k=2, max_chars_per_block=600,
)
with open("sample_transcripts.md", "w") as f:
    f.write(transcripts_md)
print(f"Wrote sample_transcripts.md ({len(transcripts_md)} chars)")

display(Markdown(transcripts_md))

## 15 · Per-task reward curves (from `trainer.state.log_history`)

Live Trackio is the primary instrument; this is the static fallback so the artifact survives the Colab session ending. Each curve is a single task's reward function — `0.0` rows (rows from other tasks) are masked before smoothing.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_df = pd.DataFrame(trainer.state.log_history)
print("Reward-related columns:", [c for c in log_df.columns if "reward" in c])

fig, ax = plt.subplots(figsize=(12, 5))
for task in TASK_LIST:
    candidates = [c for c in log_df.columns
                  if task in c and "reward" in c and "std" not in c.lower() and "process" not in c and "terminal" not in c]
    if not candidates:
        continue
    col = candidates[0]
    subset = log_df[["step", col]].dropna()
    subset = subset[subset[col] > 0.0].reset_index(drop=True)
    if subset.empty:
        continue
    smooth = subset[col].rolling(window=10, min_periods=1).mean()
    ax.plot(subset["step"], smooth, linewidth=2.0, label=task)

ax.set_xlabel("GRPO step")
ax.set_ylabel("Reward (10-step EMA, gated rows only)")
ax.set_title("Per-task reward curves — A100 demo run")
ax.legend(loc="best", fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=140)
plt.show()

## 16 · Save LoRA + (optional) Hub push + teardown

In [ ]:
model.save_pretrained("grpo_bioresearch_a100_lora")
tokenizer.save_pretrained("grpo_bioresearch_a100_lora")
print("LoRA saved to grpo_bioresearch_a100_lora/")

# Optional: push to the Hub if HF_TOKEN is set in the env.
HF_REPO = os.environ.get("HF_PUSH_REPO")
if HF_REPO and os.environ.get("HF_TOKEN"):
    try:
        model.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
        tokenizer.push_to_hub(HF_REPO, token=os.environ["HF_TOKEN"])
        print(f"Pushed to https://huggingface.co/{HF_REPO}")
    except Exception as e:
        print(f"Hub push failed ({type(e).__name__}: {e}); LoRA still saved locally.")
else:
    print("Skipped Hub push (set HF_PUSH_REPO + HF_TOKEN to enable).")

server_proc.terminate()
try:
    server_proc.wait(timeout=5)
except Exception:
    server_proc.kill()
print("Server terminated")